Add to data frame: total sentences, FOG, total complex words, total future sentences, future version of prior data frame

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install syllables
import nltk
nltk.download('punkt_tab')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.1
    Uninstalling importlib_metadata-9.0.1:
      Successfully uninstalled importlib_metadata-9.0.1


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import os
import numpy as np
import pandas as pd
import syllables
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from bs4 import BeautifulSoup

In [ ]:
cmufile = "/content/drive/MyDrive/Fall 2026/Text Mining/Fall 2026 Text Mining/week 04/cmudict.txt"


with open(cmufile, encoding = 'latin-1') as cmu:
  cmuLines = cmu.readlines()

syllableDict = {}
for aline in cmuLines[56:]:
  syllableCount =0
  for char in aline:
    if char.isdigit():
      syllableCount +=1
  syllableDict[aline.split()[0].lower()] = syllableCount

def getSyllables(word):
  syllables = syllableDict[word]
  return syllables

def isComplexWord(word):
  syllables = syllableDict[word]
  if syllables > 2:
    return True
  else:
    return False

def isFuture(sentence):
    futureWords = ['will', 'shall', 'may', 'might', 'should', 'going']
    sentWords = word_tokenize(sentence.lower())

    for word in futureWords:
        if word in sentWords:
            return True

    if 'gon' in sentWords and 'na' in sentWords:
        return True
    if 'going' in sentWords and 'to' in sentWords:
        return True

    return False



In [ ]:

folder_path = "/content/drive/MyDrive/Fall 2026/Text Mining/10-Ks Assignment"
rows = []


In [ ]:
print (os.listdir(folder_path))

['10-K_1770787_2025-02-13.txt', '10-K_1690080_2025-03-31.txt', '10-K_1599407_2025-03-31.txt', '10-K_34782_2025-02-18.txt', '10-K_1347858_2025-03-20.txt', '10-K_1950976_2025-03-03.txt', '10-K_1860782_2025-03-25.txt', '10-K_1890702_2025-03-07.txt', '10-K_1948175_2025-03-10.txt', '10-K_933972_2025-03-28.txt', '10-K_910638_2025-03-27.txt', '10-K_66740_2025-02-05.txt', '10-K_1650648_2025-02-28.txt', '10-K_1946563_2025-03-27.txt', '10-K_1785173_2025-02-27.txt', '10-K_1555279_2025-03-07.txt', '10-K_1580808_2025-02-25.txt', '10-K_824142_2025-02-27.txt', '10-K_1876255_2025-03-26.txt', '10-K_1634452_2025-03-31.txt', '10-K_1982701_2025-03-31.txt', '10-K_1800_2025-02-21.txt', '10-K_1018840_2025-03-31.txt', '10-K_934549_2025-03-17.txt', '10-K_1070494_2025-02-27.txt', '10-K_899629_2025-02-14.txt', '10-K_843006_2025-03-25.txt', '10-K_1962918_2025-03-19.txt', '10-K_949858_2025-03-11.txt', '10-K_935036_2025-02-27.txt', '10-K_1680062_2025-03-03.txt', '10-K_2098_2025-03-06.txt', '10-K_715579_2025-03-14.t

In [ ]:
print ("Number of files:", len(os.listdir(folder_path)))

Number of files: 100


In [ ]:
#Loading Dictionary into Sets
def load_dictionary_set(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return set(f.read().lower().split())

#Folder with my dictionaries from drive
dictionary_folder = "/content/drive/MyDrive/Fall 2026/Text Mining/Dictionaries/"
negative_words = load_dictionary_set(dictionary_folder + "negative words.txt")
positive_words = load_dictionary_set(dictionary_folder + "positive words.txt")
litigious_words = load_dictionary_set(dictionary_folder + "litigious words.txt")

for filename in os.listdir(folder_path):
  if filename.endswith(".txt"):
    filepath = os.path.join (folder_path, filename)

# Read every file
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
      raw_html = f.read()

#clean the text
      clean_text = BeautifulSoup(raw_html,'lxml').get_text()
      sentences = sent_tokenize(clean_text)
      total_sentences = len(sentences)
      future_sentences = []
      for sentence in sentences:
        if isFuture(sentence):
            future_sentences.append(sentence)
      total_future_sentences = len(future_sentences)

      words = word_tokenize(clean_text.lower())
      words = [word for word in words if word.isalpha()]
      word_count = len(words)

# Count  occurrences
      pos_count = sum(1 for w in words if w in positive_words)
      neg_count = sum(1 for w in words if w in negative_words)
      lit_count = sum(1 for w in words if w in litigious_words)

# Calculate rates
      pos_rate = pos_count / word_count if word_count > 0 else 0
      neg_rate = neg_count / word_count if word_count > 0 else 0
      lit_rate = lit_count / word_count if word_count > 0 else 0

      future_text = " ".join(future_sentences)
      future_words = word_tokenize(future_text.lower())
      future_words = [word for word in future_words if word.isalpha()]
      future_word_count = len(future_words)

      future_pos_count = sum(1 for w in future_words if w in positive_words)
      future_neg_count = sum(1 for w in future_words if w in negative_words)
      future_lit_count = sum(1 for w in future_words if w in litigious_words)

      future_pos_rate = future_pos_count / future_word_count if future_word_count > 0 else 0
      future_neg_rate = future_neg_count / future_word_count if future_word_count > 0 else 0
      future_lit_rate = future_lit_count / future_word_count if future_word_count > 0 else 0

      complex_count = 0
      for word in words:
        if word in syllableDict:
          if isComplexWord(word):
            complex_count += 1

      future_complex_count = 0
      for word in future_words:
        if word in syllableDict:
          if isComplexWord(word):
            future_complex_count += 1
      if total_sentences > 0 and word_count > 0:
        fog = 0.4 * ((word_count / total_sentences) + 100 * (complex_count / word_count))
      else:
        fog = 0

      if total_future_sentences > 0 and future_word_count > 0:
        future_fog = 0.4 * ((future_word_count / total_future_sentences) + 100 * (future_complex_count / future_word_count))
      else:
        future_fog = 0


      row = {
        'Filename': filename,
        'TotalSentences': total_sentences,
        'WordCount': word_count,
        'PositiveWordCount': pos_count,
        'NegativeWordCount': neg_count,
        'LitigiousWordCount': lit_count,
        'PositivityRate': pos_rate,
        'NegativityRate': neg_rate,
        'LitigiousRate': lit_rate,
        'FuturePositivityRate': future_pos_rate,
        'FutureNegativeRate': future_neg_rate,
        'FutureLitigiousRate': future_lit_rate,
        'ComplexWordCount': complex_count,
        'FOG': fog,
        'FutureTotalSentences': total_future_sentences,
        'FutureWordCount': future_word_count,
        'FutureComplexWordCount': future_complex_count,
        'FutureFOG': future_fog
      }
      rows.append(row)

df=pd.DataFrame(rows)
pd.set_option('display.max_rows', None)
display(df)


,Filename,TotalSentences,WordCount,PositiveWordCount,NegativeWordCount,LitigiousWordCount,PositivityRate,NegativityRate,LitigiousRate,FuturePositivityRate,FutureNegativeRate,FutureLitigiousRate,ComplexWordCount,FOG,FutureTotalSentences,FutureWordCount,FutureComplexWordCount,FutureFOG
0,10-K_1770787_2025-02-13.txt,47176,588002,770,4729,5594,0.001310,0.008042,0.009514,0.006300,0.030810,0.019065,67828,9.599737,924,54918,12673,33.004515
1,10-K_1690080_2025-03-31.txt,15011,387953,930,5317,6401,0.002397,0.013705,0.016499,0.005026,0.025787,0.012668,79849,18.570685,1396,82173,18851,32.721522
2,10-K_1599407_2025-03-31.txt,15473,451910,985,5132,6784,0.002180,0.011356,0.015012,0.003531,0.019464,0.011796,96237,20.200789,1359,102239,22945,39.069426
3,10-K_34782_2025-02-18.txt,20717,536934,646,4291,6282,0.001203,0.007992,0.011700,0.002558,0.014319,0.005528,86262,16.793287,425,34011,5950,39.008089
4,10-K_1347858_2025-03-20.txt,9783,362287,627,4911,6197,0.001731,0.013556,0.017105,0.003915,0.019531,0.017028,79271,23.565208,547,43163,9522,40.387661
5,10-K_1950976_2025-03-03.txt,10439,323961,917,3059,4345,0.002831,0.009442,0.013412,0.005537,0.016300,0.007312,62717,20.157260,1333,70430,15517,29.947005
6,10-K_1860782_2025-03-25.txt,10715,361240,2208,4921,6494,0.006112,0.013623,0.017977,0.008762,0.022722,0.020564,81960,22.560801,1287,69052,17495,31.595775
7,10-K_1890702_2025-03-07.txt,1151,106998,49,441,640,0.000458,0.004122,0.005981,0.000000,0.004198,0.003265,36373,50.781999,26,6431,2751,116.049331
8,10-K_1948175_2025-03-10.txt,1084,92176,44,300,501,0.000477,0.003255,0.005435,0.000154,0.002918,0.001689,30463,47.232777,24,6512,2866,126.137756
9,10-K_933972_2025-03-28.txt,64995,710345,845,2398,4036,0.001190,0.003376,0.005682,0.006392,0.024090,0.012437,54482,7.439608,737,43338,9549,32.334815


Gonna tokenized to gon na

In [ ]:
text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy!"

sentences = sent_tokenize(text)
sentWords = word_tokenize(sentences[0])

print(sentWords)

['NLTK', 'is', 'gon', 'na', 'be', 'a', 'great', 'NLP', 'toolkit', '.']


In [ ]:
#print(getSyllables('kevin'))
#print(syllables.estimate('kevinfdsafdafdfdsfa'))

#print (isComplexWord('kevin'))

text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy!"
sentences = sent_tokenize(text)
words = word_tokenize(sentences[0])
#print(words)
print(sentences[0])
print(isFuture(sentences[0]))


NLTK is gonna be a great NLP toolkit.
True
